In [115]:
##load tools
import xarray as xr
import numpy as np
import cmocean as cmo
import cartopy
import cartopy.crs as ccrs
import pandas as pd
import matplotlib.pyplot as plt
import gsw 
from argopy import ArgoFloat

In [116]:
#open float data
WMOs=['1902371','2903938'] #pick float number
with argopy.set_options(mode='expert'):
    papafloat = ArgoFloat(WMOs[0]).open_dataset('Sprof')
    tzfloat = ArgoFloat(WMOs[1]).open_dataset('Sprof')

In [117]:
display(tzfloat)

<xarray.Dataset> Size: 75MB
Dimensions:                            (N_PROF: 110, N_PARAM: 14, N_CALIB: 1,
                                        N_LEVELS: 1422)
Dimensions without coordinates: N_PROF, N_PARAM, N_CALIB, N_LEVELS
Data variables: (12/128)
    DATA_TYPE                          <U32 128B ...
    FORMAT_VERSION                     <U4 16B ...
    HANDBOOK_VERSION                   <U4 16B ...
    REFERENCE_DATE_TIME                datetime64[ns] 8B ...
    DATE_CREATION                      datetime64[ns] 8B ...
    DATE_UPDATE                        datetime64[ns] 8B ...
    ...                                 ...
    NITRATE                            (N_PROF, N_LEVELS) float32 626kB ...
    NITRATE_QC                         (N_PROF, N_LEVELS) int64 1MB ...
    NITRATE_dPRES                      (N_PROF, N_LEVELS) float32 626kB ...
    NITRATE_ADJUSTED                   (N_PROF, N_LEVELS) float32 626kB ...
    NITRATE_ADJUSTED_QC                (N_PROF, N_LEVELS) int64 1MB ...
    NITRATE_ADJUSTED_ERROR             (N_PROF, N_LEVELS) float32 626kB ...
Attributes:
    title:                Argo float vertical profile
    institution:          AOML
    source:               Argo float
    history:              2026-08-11T22:05:00Z creation (software version 1.2...
    references:           http://www.argodatamgt.org/Documentation
    user_manual_version:  1.0
    Conventions:          Argo-3.1 CF-1.6
    featureType:          trajectoryProfile
    software_version:     1.22 (version 12.05.2025 for ARGO_simplified_profile)
    id:                   https://doi.org/10.17882/42182

In [ ]:
fig = plt.figure(figsize=(8,8),layout='tight')
ax = fig.add_subplot(111, projection=ccrs.PlateCarree())
ax.add_feature(cartopy.feature.LAND, edgecolor='None', facecolor='darkgray', zorder=2)
ax.add_feature(cartopy.feature.COASTLINE, edgecolor='k', zorder=2)
ax.set_extent([papafloat.LONGITUDE.min()-5, papafloat.LONGITUDE.max()+25, papafloat.LATITUDE.min()-10, papafloat.LATITUDE.max()+10])
d=ax.scatter(papafloat.LONGITUDE,papafloat.LATITUDE,c='r',label='float 1902371')
b=ax.scatter(tzfloat.LONGITUDE,tzfloat.LATITUDE,c='green',label='float 2903938')
ax.scatter(-144.9,50.1,marker='*',s=400,facecolor='k',label='Ocean Station Papa',edgecolor='k',zorder=5)
ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,linewidth=2, color='gray', alpha=0.1, linestyle='--')
ax.legend()

In [ ]:
fig,ax=plt.subplots(1,2,sharex=True,sharey=True)
# ax[0].set_ylim(-200,0)
for b in papafloat.N_PROF:
    prof=papafloat.isel(N_PROF=b)
    ax[0].scatter(prof.DOWN_IRRADIANCE490,-prof.PRES,)
    prof=tzfloat.isel(N_PROF=b)
    ax[1].scatter(prof.DOWN_IRRADIANCE490,-prof.PRES,)

In [ ]:
fig,ax=plt.subplots(1,2,sharex=True,sharey=True)
# ax[0].set_ylim(-200,0)
for b in papafloat.N_PROF:
    prof=papafloat.isel(N_PROF=b)
    ax[0].scatter(prof.CHLA_ADJUSTED,-prof.PRES,)
    prof=tzfloat.isel(N_PROF=b)
    ax[1].scatter(prof.CHLA_ADJUSTED,-prof.PRES,)

In [ ]:
#calculate MLD for profiles
def mld_calc(floatdata):
    MLD=np.zeros(len(floatdata.N_PROF))
    MLD[:]=np.nan
    for i in floatdata.N_PROF:
        prof=floatdata.isel(N_PROF=i)
        densprof=gsw.density.sigma0(prof.PSAL,prof.TEMP).values
        refdens=densprof[np.abs(prof.PRES.values - 10).argmin()]
        refdens=densprof[10]
        for g in range(0,len(densprof)):
            dens=densprof[g]
            densdiff=np.abs(refdens-dens)
            if densdiff>0.03:
                MLD[i]=prof.isel(N_LEVELS=g).PRES
                break
    return(MLD)

In [ ]:
print(mld_calc(papafloat))